In [1]:
import polars as pl
import pandas as pd
import boto3

In [2]:
from tqdm import tqdm
import os

In [3]:
from collections import defaultdict

In [4]:
from dap_prinz_green_jobs.getters.data_getters import get_s3_data_paths

In [5]:
data_file_names = get_s3_data_paths("open-jobs-lake", "job_quality/welsh_analysis/quality_extractions_green_sectors_interim/", file_types=["*.parquet"])

2025-03-27 11:51:51,606 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


In [6]:
len(data_file_names)

100

In [10]:
all_wales_data_file_names = get_s3_data_paths(
    "open-jobs-lake", "job_quality/welsh_analysis/quality_extractions_interim/",
    file_types=["*.parquet"])
len(all_wales_data_file_names)

100

In [7]:
job_advert_s3_location = "s3://prinz-green-jobs/outputs/data/ojo_application"
combined_green_measures_filename = os.path.join(
    job_advert_s3_location,
    "extracted_green_measures/analysis/20241121/combined_green_measures_and_meta.parquet",
)
combined_all_data_orig = pl.read_parquet(combined_green_measures_filename)

In [55]:
len(combined_all_data_orig)

5967229

In [8]:
titles_file = os.path.join(
    job_advert_s3_location,
    "deduplicated_sample/20241114/latest_update_20241114_titles.parquet",
)
titles_data = pl.read_parquet(titles_file)

In [ ]:
job_descriptions = pd.read_parquet("s3://open-jobs-lake/job_quality/welsh_analysis/subset_green_sectors_descriptions.parquet")


In [19]:
len(job_descriptions)

47551

## Read in job quality data

In [11]:
quality_by_ids = defaultdict(list)
for data_file_name in tqdm(data_file_names):
    jq_df_filtered_chunk = pd.read_parquet(f"s3://open-jobs-lake/" + data_file_name,
                                          )
    one_file_dict = jq_df_filtered_chunk.groupby('id').agg({'subcategory': 'unique'}).to_dict(orient='dict')['subcategory']
    for subcat, id_list in one_file_dict.items():
        quality_by_ids[subcat] += list(id_list)

quality_by_ids = {k:list(set(v)) for k, v in quality_by_ids.items()}


  0%|                                                   | 0/100 [00:00<?, ?it/s]

2025-03-27 11:55:15,221 - botocore.credentials - INFO - Found credentials in shared credentials file: ~/.aws/credentials


100%|█████████████████████████████████████████| 100/100 [00:45<00:00,  2.20it/s]


In [12]:
# Should be 47551 total job adverts from green sectors
len(quality_by_ids) # Not all adverts will have job quality measures?

39023

In [17]:
quality_df = pd.DataFrame([quality_by_ids]).T
quality_df = pd.get_dummies(quality_df.explode(column=0), prefix=None).groupby(level=0).sum()
quality_df.rename(columns = {c: c.split('0_')[1] for c in quality_df.columns}, inplace=True) # prefixes them all with 0_ for some reason
quality_df = quality_df.reset_index().rename(columns={"index": "id"})
quality_df.head(2)

,id,AUTONOMY,CAREER,CARING,COMP,CONTRACT,DISABILITY,FLEX_HOURS,FLEX_LOC,HEALTH,...,LOC,MISC,M_HEALTH,PERKS,REWARD,SENSE OF PURPOSE,SHIFT,SOCIAL,SPONSORSHIP,VOICE REPRESENTATION
0,41547788,0,0,0,1,1,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
1,41548335,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
# Add rows for job ids which didn't have quality found but were in the sample
no_quality_ids = set(job_descriptions['id']).difference(set(quality_by_ids.keys()))
quality_df = pd.concat([quality_df, pd.DataFrame({'id': list(no_quality_ids)})])

8528

## Add green measures + extra info to job quality

In [32]:
subset_combined_all_data_orig = combined_all_data_orig.filter(pl.col('job_id').is_in(quality_df['id'].tolist())).to_pandas()

In [33]:
subset_titles_data = titles_data[['id', 'parent_sector', 'sector']].filter(pl.col('id').is_in(quality_df['id'].tolist())).to_pandas()

In [49]:
subset_dataset = subset_combined_all_data_orig.merge(subset_titles_data, left_on="job_id", right_on="id")
subset_dataset = subset_dataset.merge(quality_df, left_on="job_id", right_on="id")
subset_dataset.drop(['id_x', 'id_y', '__index_level_0__'], axis=1, inplace=True)

/Users/elizabethgallagher/miniconda3/envs/dap_prinz_green_jobs/lib/python3.8/site-packages/pandas/core/frame.py:9190: FutureWarning: Passing 'suffixes' which cause duplicate columns {'id_x'} in the result is deprecated and will raise a MergeError in a future version.
  return merge(


In [50]:
subset_dataset['sector'].nunique()

12

In [51]:
subset_dataset['itl_1_name'].nunique()

12

In [52]:
subset_dataset['job_id'].nunique()

47551

In [53]:
subset_dataset.to_parquet("s3://open-jobs-lake/job_quality/welsh_analysis/subset_green_sectors_quality_combined.parquet")

# Totals
- Total number of job adverts per country
- Total number of job adverts per sector
- Total number of job adverts per country and sector

In [63]:
itl_ads = combined_all_data_orig[["job_id", "itl_1_name"]].to_pandas()
itl_ads["country"] = itl_ads['itl_1_name'].apply(
    lambda x: x if x in ["Wales", "Scotland", None] else "England")
itl_ads["country"].value_counts()

England     5458040
Scotland     197598
Wales        139337
Name: country, dtype: int64

In [74]:
counts_dataset = itl_ads.merge(titles_data[['id', 'sector']].to_pandas(), left_on="job_id", right_on="id")
counts_dataset = counts_dataset.groupby(["country", "sector"], dropna=False)['job_id'].count().reset_index()

In [75]:
counts_dataset.groupby('country', dropna=False)["job_id"].sum()

country
England     5458040
Scotland     197598
Wales        139337
NaN          172254
Name: job_id, dtype: int64

In [78]:
counts_dataset.to_parquet("s3://open-jobs-lake/job_quality/welsh_analysis/country_sectors_counts.parquet")